# Qwen LoRA SFT / Data Pipeline / DPO Learning Notebook

这个 notebook 是本项目的“可操作学习地图”。它把命令行脚本串成可以逐格运行的流程，帮助理解：

- Qwen base 推理是什么样子。
- LoRA SFT 怎样准备数据、训练 adapter、加载 adapter。
- 为什么公开通用数据集能跑通工程闭环，但不一定修正 LoRA/SFT/DPO 技术概念误解。
- Stage 2B 如何做自采集数据、清洗、转换。
- Stage 3B 如何训练 custom LoRA SFT。
- Stage 4B 如何做 base vs public-SFT vs custom-SFT 三方对比。
- Stage 2B.2/2B.3 如何做 badcase patch，并用固定 prompt 当作回归测试。
- Stage 5 为什么要先 tiny DPO，再根据显存和行为决定是否扩大。

当前 notebook 覆盖到 Stage 5P：naive DPO v6、Stage 5H prompt 7 数据/eval 设计、expanded behavior gate、DPO v7/v8 以及多轮 SFT repair probes。多轮运行证明硬件可跑，loss 和 preference accuracy 也能好看，但 prompt 7 行为 gate 仍是核心缺口。


## 0. 使用方式

建议用 `qwen-lora-local` 环境打开这个 notebook。大多数 cell 是轻量检查；训练 cell 默认不会执行，需要手动把开关改成 `True`。

如果你只是复盘项目，按顺序运行环境检查、推理、报告读取即可。如果你要重新训练，再打开训练开关。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

os.environ["HF_HOME"] = str((PROJECT_ROOT / ".hf_cache").resolve())
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("HF_HOME:", os.environ["HF_HOME"])

def run_cmd(args, timeout=None):
    print("$", " ".join(str(x) for x in args))
    result = subprocess.run(
        [str(x) for x in args],
        cwd=PROJECT_ROOT,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        timeout=timeout,
    )
    print(result.stdout)
    if result.stderr:
        print("[stderr]")
        print(result.stderr)
    result.check_returncode()
    return result

## 1. 环境检查

先确认 Python 包版本和 CUDA。这个项目之前在 Windows 原生环境里遇到过高版本 Hugging Face 栈导致 `python.exe` 进程级崩溃的问题，所以环境版本是项目经验的一部分。

In [ ]:
run_cmd([sys.executable, "scripts/check_env.py"], timeout=60)
run_cmd([
    sys.executable,
    "-c",
    "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')",
], timeout=60)

## 2. 先讲清楚术语

- SFT 是 supervised fine-tuning，指训练目标和数据形式。
- LoRA 是 parameter-efficient fine-tuning 方法，指只训练 adapter 小矩阵。
- LoRA SFT 的意思是：用 LoRA adapter 来做 SFT。
- DPO 是 direct preference optimization，使用 chosen/rejected 偏好对进一步调模型行为。

Stage 4A 的重要发现是：公开通用数据集虽然跑通了 LoRA SFT，但没有修正这些技术概念误解。

## 3. Base 模型推理

先看没有 adapter 的 Qwen 怎么回答。这是后续对比的 baseline。

In [ ]:
run_cmd([
    sys.executable,
    "scripts/infer.py",
    "--prompt", "请用三点解释什么是LoRA微调。",
    "--max_new_tokens", "96",
    "--local_files_only",
], timeout=180)

## 4. Stage 2A：公开数据集准备

当前公开数据集已经准备好：

- raw: `data/raw/alpaca_gpt4_data_zh_1200.jsonl`
- train: `data/processed/sft_train.jsonl`
- eval: `data/processed/sft_eval.jsonl`

如果以后要重跑，把下面的 `RUN_PUBLIC_DATA_PREP` 改成 `True`。

In [ ]:
RUN_PUBLIC_DATA_PREP = False

if RUN_PUBLIC_DATA_PREP:
    run_cmd([
        sys.executable,
        "scripts/download_hf_sft_data.py",
        "--dataset_name", "llm-wizard/alpaca-gpt4-data-zh",
        "--split", "train",
        "--output_file", "data/raw/alpaca_gpt4_data_zh_1200.jsonl",
        "--max_samples", "1200",
        "--seed", "42",
    ], timeout=600)
    run_cmd([
        sys.executable,
        "scripts/prepare_data.py",
        "--input_file", "data/raw/alpaca_gpt4_data_zh_1200.jsonl",
        "--train_file", "data/processed/sft_train.jsonl",
        "--eval_file", "data/processed/sft_eval.jsonl",
        "--eval_ratio", "0.1",
        "--max_samples", "1200",
        "--min_answer_chars", "20",
        "--seed", "42",
    ], timeout=120)
else:
    for path in ["data/processed/sft_train.jsonl", "data/processed/sft_eval.jsonl"]:
        rows = [line for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
        print(path, "rows =", len(rows))

## 5. Stage 3A：公开数据 LoRA SFT

这一步训练耗时约 24 分钟，训练时显存观察约 `5.5GB / 8GB`。默认不执行。

已经完成的 adapter：`outputs/sft_lora_qwen05b_public`。

In [ ]:
RUN_PUBLIC_SFT_TRAINING = False

if RUN_PUBLIC_SFT_TRAINING:
    run_cmd([
        sys.executable,
        "scripts/train_sft_lora.py",
        "--train_file", "data/processed/sft_train.jsonl",
        "--eval_file", "data/processed/sft_eval.jsonl",
        "--output_dir", "outputs/sft_lora_qwen05b_public",
        "--max_length", "512",
        "--batch_size", "1",
        "--grad_accum", "8",
        "--epochs", "1",
        "--logging_steps", "10",
        "--eval_steps", "100",
        "--save_steps", "100",
        "--report_to", "none",
        "--local_files_only",
    ], timeout=7200)
else:
    adapter_dir = Path("outputs/sft_lora_qwen05b_public")
    print("Adapter exists:", adapter_dir.exists())
    print("Adapter files:", [p.name for p in adapter_dir.glob("adapter_*")])

## 6. Stage 4A：Base vs Public-SFT 对比

这一步是现在最重要的结论来源：公开数据 SFT 能训练成功，但没有修正 LoRA/SFT/DPO 的概念误解。

In [ ]:
RUN_STAGE4A_COMPARE = False

if RUN_STAGE4A_COMPARE:
    run_cmd([
        sys.executable,
        "scripts/compare_outputs.py",
        "--prompt_file", "data/samples/smoke_prompts.jsonl",
        "--adapter_path", "outputs/sft_lora_qwen05b_public",
        "--output_file", "reports/compare_outputs_public_sft.jsonl",
        "--max_new_tokens", "96",
        "--local_files_only",
    ], timeout=900)

rows = [json.loads(line) for line in Path("reports/compare_outputs_public_sft.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
for row in rows:
    print("=" * 80)
    print("Prompt:", row["prompt"])
    print("\nBase:\n", row["base_answer"][:500])
    print("\nPublic-SFT:\n", row["sft_answer"][:500])

## 7. Stage 4A 结论

公开通用数据集的价值：

- 证明了数据转换、tokenizer、LoRA 训练、adapter 保存/加载都能工作。
- 得到了一个可比较的 public-SFT baseline。

公开通用数据集的不足：

- 没有修正 LoRA/SFT/DPO 的目标技术概念。
- LoRA 仍可能被解释成通信 LoRa。
- SFT/DPO 仍可能被解释成无关缩写。

所以 Stage 2B 不是锦上添花，而是必要步骤：我们需要自采集技术数据来教模型目标概念和项目叙述方式。

## 8. Stage 2B：自采集技术数据

Stage 2B 已经完成并经过多轮修订。第一版生成 160 条 seed 样本，但训练后仍出现幻觉，说明数据里泛化的项目总结样本太多。

后续做了三次数据侧调整：

- 把 `project_record_summary` 从 60 条降到 20 条，减少模板化项目记录对模型的干扰。
- Stage 2B.1/2B 修订新增 targeted QA，直接覆盖 LoRA/SFT/DPO、数据清洗、DPO 显存、loss vs behavior 等固定 prompt。
- Stage 2B.2 新增 8 条 badcase patch，专门补 public-SFT motivation 和 loss-vs-behavior。
- Stage 2B.3 新增 10 条 loss-vs-behavior 样本和 7 条 replay 样本，并额外导出 focused patch train 文件。

当前产物：

- `data/raw/custom_sources.jsonl`：10 个项目自有技术来源。
- `data/raw/custom_cleaned_chunks.jsonl`：96 个清洗后 chunk。
- `data/raw/custom_instruction_seed.jsonl`：157 条 instruction-answer seed。
- `data/processed/custom_sft_train.jsonl`：142 条 train。
- `data/processed/custom_sft_eval.jsonl`：15 条 eval。
- `data/processed/custom_sft_stage2b3_patch_train.jsonl`：28 条 focused patch train。


In [ ]:
RUN_STAGE2B_PREP = False

if RUN_STAGE2B_PREP:
    run_cmd([
        sys.executable,
        "scripts/prepare_custom_technical_data.py",
        "--raw_sources_file", "data/raw/custom_sources.jsonl",
        "--cleaned_chunks_file", "data/raw/custom_cleaned_chunks.jsonl",
        "--instruction_seed_file", "data/raw/custom_instruction_seed.jsonl",
        "--train_file", "data/processed/custom_sft_train.jsonl",
        "--eval_file", "data/processed/custom_sft_eval.jsonl",
        "--eval_ratio", "0.1",
        "--max_doc_samples", "20",
        "--seed", "42",
    ], timeout=300)

for path in [
    "data/raw/custom_sources.jsonl",
    "data/raw/custom_cleaned_chunks.jsonl",
    "data/raw/custom_instruction_seed.jsonl",
    "data/processed/custom_sft_train.jsonl",
    "data/processed/custom_sft_eval.jsonl",
]:
    rows = [line for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
    print(path, "rows =", len(rows))


## 9. Stage 2B 校验

Stage 2B 的校验重点：

- train/eval 都是 Qwen chat JSONL。
- role 顺序是 `system -> user -> assistant`。
- prompt 不重复。
- 原始 token 长度不超过 512，避免训练时截断。
- 能被 `ChatSFTDataset` 正常编码。

In [ ]:
for path in ["data/processed/custom_sft_train.jsonl", "data/processed/custom_sft_eval.jsonl"]:
    rows = [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
    roles_ok = all([m["role"] for m in row["messages"]] == ["system", "user", "assistant"] for row in rows)
    prompts = [row["messages"][1]["content"] for row in rows]
    print(path)
    print("rows:", len(rows))
    print("roles_ok:", roles_ok)
    print("unique_prompts:", len(set(prompts)))
    print("sample prompt:", prompts[0][:120].replace("\n", " "))
    print("sample answer:", rows[0]["messages"][2]["content"][:160].replace("\n", " "))
    print()

## 10. Stage 3B：custom-data LoRA SFT

Stage 3B v1 已经完成，输出 adapter：

```text
outputs/sft_lora_qwen05b_custom
```

v1 训练结果摘要：

- Train rows: 119
- Eval rows: 13
- Trainable params: 4,399,104，约 0.8826%
- Runtime: 约 12.3 分钟
- Final train loss: 0.4656
- Best observed eval loss: 0.8311，约在 epoch 5

一个很重要的观察：10 epoch 最终能让 target behavior 更明显，但 eval loss 在 epoch 5 后略微上升，说明小数据下有过拟合风险。

Stage 2B.2 后又测试了 v2/v3。结论是：小补丁从头训练 v2 会回归；从 v1 低学习率续训得到的 `outputs/sft_lora_qwen05b_custom_v3_from_v1_patch` 更稳。

默认不执行训练；如果你要复跑，把对应代码 cell 里的开关改成 `True`。


In [ ]:
RUN_CUSTOM_SFT_TRAINING = False

if RUN_CUSTOM_SFT_TRAINING:
    run_cmd([
        sys.executable,
        "scripts/train_sft_lora.py",
        "--train_file", "data/processed/custom_sft_train.jsonl",
        "--eval_file", "data/processed/custom_sft_eval.jsonl",
        "--output_dir", "outputs/sft_lora_qwen05b_custom",
        "--max_length", "512",
        "--batch_size", "1",
        "--grad_accum", "4",
        "--epochs", "10",
        "--logging_steps", "10",
        "--eval_steps", "50",
        "--save_steps", "50",
        "--report_to", "none",
        "--local_files_only",
    ], timeout=7200)
else:
    print("Custom SFT training is skipped. Set RUN_CUSTOM_SFT_TRAINING=True when ready.")
    print("Current local output: outputs/sft_lora_qwen05b_custom")
    print("Report: reports/stage3b_custom_lora_sft_report.md")


## 11. Stage 4B：Base vs Public-SFT vs Custom-SFT 三方对比

Stage 4B v1 已经完成。对比文件：

```text
reports/compare_outputs_three_way_custom.jsonl
reports/stage4b_three_way_comparison_report.md
```

v1 结果不是“全胜”，而是更真实也更有价值：custom-SFT 明显改善 6/8 个固定技术 prompt，同时留下 2 个 badcase，刚好指导了 Stage 2B.2 小数据补丁。

Stage 2B.2 后的更新结论：

- v2 从头训练 5 epoch：回归，不推荐。
- v2 从头训练 10 epoch 并选 best-eval checkpoint：仍回归，不推荐。
- v3 从 v1 低学习率续训 2 epoch：保留或改善 7/8 个固定 prompt，当前最好。

现在仍需补数据的方向：

- 为什么不能只看 loss 判断 SFT 是否成功。

这个结论很适合面试讲：我们不是只看 loss，也不是盲目加数据，而是把固定 prompt 当作回归测试集。


In [ ]:
RUN_STAGE4B_COMPARE = False

if RUN_STAGE4B_COMPARE:
    run_cmd([
        sys.executable,
        "scripts/compare_three_outputs.py",
        "--prompt_file", "data/samples/custom_technical_prompts.jsonl",
        "--public_adapter_path", "outputs/sft_lora_qwen05b_public",
        "--custom_adapter_path", "outputs/sft_lora_qwen05b_custom",
        "--output_file", "reports/compare_outputs_three_way_custom.jsonl",
        "--max_new_tokens", "128",
        "--temperature", "0",
        "--local_files_only",
    ], timeout=7200)
else:
    print("Stage 4B comparison is skipped. Set RUN_STAGE4B_COMPARE=True to rerun.")

comparison_path = Path("reports/compare_outputs_three_way_custom.jsonl")
if comparison_path.exists():
    rows = [json.loads(line) for line in comparison_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print("comparison rows:", len(rows))
    for i, row in enumerate(rows, start=1):
        print(f"{i}. prompt:", row["prompt"][:80])
        print("   custom:", row["custom_sft_answer"][:120].replace("\n", " "))


## 12. Stage 2B.2：badcase patch 与回归评测

Stage 2B.2 的结论是：v2 从头训练不稳，v3 从 v1 低学习率续训最稳。

关键发现：

- v2 从头训练 5 epoch：回归，不推荐。
- v2 从头训练 10 epoch 并选 best-eval checkpoint：仍回归，不推荐。
- v3 从 v1 低学习率续训 2 epoch：保留或改善 7/8 个固定 prompt，当前仍是最好 SFT adapter。

当前推荐 adapter：

```text
outputs/sft_lora_qwen05b_custom_v3_from_v1_patch
```

Stage 2B.2 剩下的主要 badcase 是：为什么不能只看 loss 判断 SFT 是否成功。这个问题后来进入 Stage 2B.3 gate。


In [ ]:
RUN_STAGE2B2_PREP = False
RUN_STAGE3B2_CONTINUE_FROM_V1 = False
RUN_STAGE4B2_COMPARE = False

if RUN_STAGE2B2_PREP:
    run_cmd([
        sys.executable,
        "scripts/prepare_custom_technical_data.py",
        "--raw_sources_file", "data/raw/custom_sources.jsonl",
        "--cleaned_chunks_file", "data/raw/custom_cleaned_chunks.jsonl",
        "--instruction_seed_file", "data/raw/custom_instruction_seed.jsonl",
        "--train_file", "data/processed/custom_sft_train.jsonl",
        "--eval_file", "data/processed/custom_sft_eval.jsonl",
        "--eval_ratio", "0.1",
        "--max_doc_samples", "20",
        "--seed", "42",
    ], timeout=300)

if RUN_STAGE3B2_CONTINUE_FROM_V1:
    run_cmd([
        sys.executable,
        "scripts/train_sft_lora.py",
        "--init_adapter_path", "outputs/sft_lora_qwen05b_custom",
        "--train_file", "data/processed/custom_sft_train.jsonl",
        "--eval_file", "data/processed/custom_sft_eval.jsonl",
        "--output_dir", "outputs/sft_lora_qwen05b_custom_v3_from_v1_patch",
        "--max_length", "512",
        "--batch_size", "1",
        "--grad_accum", "4",
        "--epochs", "2",
        "--lr", "5e-5",
        "--logging_steps", "10",
        "--eval_steps", "30",
        "--save_steps", "30",
        "--report_to", "none",
        "--local_files_only",
    ], timeout=7200)

if RUN_STAGE4B2_COMPARE:
    run_cmd([
        sys.executable,
        "scripts/compare_three_outputs.py",
        "--prompt_file", "data/samples/custom_technical_prompts.jsonl",
        "--public_adapter_path", "outputs/sft_lora_qwen05b_public",
        "--custom_adapter_path", "outputs/sft_lora_qwen05b_custom_v3_from_v1_patch",
        "--output_file", "reports/compare_outputs_three_way_custom_v3_from_v1_patch.jsonl",
        "--max_new_tokens", "128",
        "--temperature", "0",
        "--local_files_only",
    ], timeout=7200)
else:
    print("Stage 2B.2 heavy commands are skipped. Turn on switches above to rerun.")

report_path = Path("reports/stage2b2_badcase_patch_report.md")
if report_path.exists():
    print(report_path.read_text(encoding="utf-8")[:1800])

v3_path = Path("reports/compare_outputs_three_way_custom_v3_from_v1_patch.jsonl")
if v3_path.exists():
    rows = [json.loads(line) for line in v3_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print("v3 comparison rows:", len(rows))
    for i, row in enumerate(rows, start=1):
        print(f"{i}. prompt:", row["prompt"][:80])
        print("   custom-v3:", row["custom_sft_answer"][:120].replace("
", " "))


## 13. Stage 2B.3：DPO 前 SFT 稳定性 gate

Stage 2B.3 已完成，并且按计划停在 DPO 之前。

目标不是“再训练一个一定更好的 adapter”，而是验证：能不能修好最后一个 loss-vs-behavior prompt，同时不破坏其他 7 个 prompt。

结果很真实：

- v4：完整 Stage 2B.3 数据低学习率续训，旧能力大多保留，但第 7 个 prompt 仍弱。
- v5：focused patch 修好了第 7 个 prompt，但多个旧 prompt 回归。
- v6：降低更新强度后仍然不稳。
- v7 interpolation probe：混合 v3/v5 权重，spot-check 没修好第 7 个 prompt。

所以当前结论是：`outputs/sft_lora_qwen05b_custom_v3_from_v1_patch` 仍是当前 best adapter。先不要开始 DPO，要先复盘这个 gate。


In [ ]:
RUN_STAGE2B3_PREP = False
RUN_STAGE3B3_V4_TRAIN = False
RUN_STAGE3B3_V5_TRAIN = False
RUN_STAGE4B3_COMPARE = False

if RUN_STAGE2B3_PREP:
    run_cmd([
        sys.executable,
        "scripts/prepare_custom_technical_data.py",
        "--raw_sources_file", "data/raw/custom_sources.jsonl",
        "--cleaned_chunks_file", "data/raw/custom_cleaned_chunks.jsonl",
        "--instruction_seed_file", "data/raw/custom_instruction_seed.jsonl",
        "--train_file", "data/processed/custom_sft_train.jsonl",
        "--eval_file", "data/processed/custom_sft_eval.jsonl",
        "--stage2b3_patch_train_file", "data/processed/custom_sft_stage2b3_patch_train.jsonl",
        "--stage2b3_loss_repeats", "12",
        "--eval_ratio", "0.1",
        "--max_doc_samples", "20",
        "--seed", "42",
    ], timeout=300)

if RUN_STAGE3B3_V4_TRAIN:
    run_cmd([
        sys.executable,
        "scripts/train_sft_lora.py",
        "--init_adapter_path", "outputs/sft_lora_qwen05b_custom_v3_from_v1_patch",
        "--train_file", "data/processed/custom_sft_train.jsonl",
        "--eval_file", "data/processed/custom_sft_eval.jsonl",
        "--output_dir", "outputs/sft_lora_qwen05b_custom_v4_stage2b3_loss_patch",
        "--max_length", "512",
        "--batch_size", "1",
        "--grad_accum", "4",
        "--epochs", "2",
        "--lr", "3e-5",
        "--report_to", "none",
        "--local_files_only",
    ], timeout=7200)

if RUN_STAGE3B3_V5_TRAIN:
    run_cmd([
        sys.executable,
        "scripts/train_sft_lora.py",
        "--init_adapter_path", "outputs/sft_lora_qwen05b_custom_v3_from_v1_patch",
        "--train_file", "data/processed/custom_sft_stage2b3_patch_train.jsonl",
        "--eval_file", "data/processed/custom_sft_eval.jsonl",
        "--output_dir", "outputs/sft_lora_qwen05b_custom_v5_stage2b3_focused_patch",
        "--max_length", "512",
        "--batch_size", "1",
        "--grad_accum", "1",
        "--epochs", "4",
        "--lr", "8e-5",
        "--report_to", "none",
        "--local_files_only",
    ], timeout=7200)

if RUN_STAGE4B3_COMPARE:
    run_cmd([
        sys.executable,
        "scripts/compare_three_outputs.py",
        "--prompt_file", "data/samples/custom_technical_prompts.jsonl",
        "--public_adapter_path", "outputs/sft_lora_qwen05b_public",
        "--custom_adapter_path", "outputs/sft_lora_qwen05b_custom_v4_stage2b3_loss_patch",
        "--output_file", "reports/compare_outputs_three_way_custom_v4_stage2b3_loss_patch.jsonl",
        "--max_new_tokens", "128",
        "--temperature", "0",
        "--local_files_only",
    ], timeout=7200)
else:
    print("Stage 2B.3 heavy commands are skipped. Review the report first.")

report_path = Path("reports/stage2b3_sft_stability_gate_report.md")
if report_path.exists():
    print(report_path.read_text(encoding="utf-8")[:2200])


## 14. Stage 5：DPO 拆分计划、candidate 重试和显存结论

当前观察：

- LoRA SFT 训练约 `5.5GB / 8GB`。
- Adapter 推理约 `1.2GB / 8GB`。
- Stage 3B custom LoRA SFT 能在 8GB 上完成。
- Stage 5 tiny DPO v1/v2/v3/v4/v5 和 larger naive v6 都没有 OOM，硬件不是当前 blocker。

Stage 5 从这个 adapter 开始：

```text
outputs/sft_lora_qwen05b_custom_v3_from_v1_patch
```

为什么还是 v3？因为 v4 没修好第 7 个 prompt，v5 修好了但破坏旧能力，v6 仍然不稳。后续 DPO v1/v2/v3/v4/v5 也没有通过行为 gate。v3 不是完美，但它仍是目前整体最稳的 SFT checkpoint。

Stage 5 拆分和结果：

- Stage 5A：已准备 `data/processed/dpo_tiny_train.jsonl`，共 33 条 preference pairs。
- Stage 5B：已完成 tiny DPO smoke test，输出 `outputs/dpo_lora_qwen05b_tiny`；4 个 optimizer step，约 32.8 秒，无 OOM/原生崩溃。
- Stage 5C：已完成固定 prompt 对比，结果是 5 个明确保住、1 个观察、2 个失败，行为 gate 未通过。
- Stage 5A.2/B.2/C.2：v2 preference 数据和 DPO 跑通，但 prompt 7 仍弱。
- Stage 5A.3/B.3/C.3：exact-bad-output DPO 跑通，但多个旧 prompt 回归。
- Stage 5E/B.4/C.4：candidate-derived v4，20 train / 5 eval，无 OOM，adapter 可加载，结构化评分 6/8。
- Stage 5F/B.5/C.5：focused candidate v5，28 train / 7 eval，无 OOM，adapter 可加载，结构化评分仍 6/8，loss-vs-behavior 更弱。
- Stage 5 structured scoring：用透明规则给 v1/v2/v3/v4/v5 四方输出打分，确认 SFT v3 仍优于 DPO adapters。
- Stage 5G/B.6/C.6：larger naive v6，192 train / 24 eval，单独 frozen reference model，无 OOM，结构化评分 7/8；prompt 4 修好，但 prompt 7 仍未过。
- Stage 5H：prompt 7 数据/eval 设计完成，278 train / 55 eval，新增 72 条 prompt 7 train pair 和 24 条 held-out eval pair。
- Stage 5I-5P：expanded v6、DPO v7/v8、SFT repair probes 都已完成；preference eval 可以到 1.0，但固定 behavior gate 仍能拦住 prompt 7。

关键文件：

- `scripts/prepare_tiny_dpo_data.py`
- `scripts/prepare_candidate_dpo_data.py`
- `scripts/prepare_candidate_dpo_v5_data.py`
- `scripts/prepare_larger_dpo_v6_data.py`
- `scripts/prepare_stage5h_prompt7_data.py`
- `scripts/train_dpo.py`
- `scripts/compare_four_outputs.py`
- `scripts/score_fixed_prompt_outputs.py`
- `scripts/score_expanded_behavior_outputs.py`
- `scripts/prepare_stage5n_prompt7_micro_sft_data.py`
- `reports/stage5j_to_5p_prompt7_repair_report.md`
- `reports/stage5_candidate_dpo_v4_v5_report.md`
- `reports/stage5g_naive_dpo_v6_report.md`
- `reports/stage5_structured_behavior_score_report.md`
- `reports/stage5h_prompt7_data_and_eval_design.md`

运行 tiny DPO 时要记录：专用显存峰值、共享显存增长、系统内存、step 速度、是否 OOM/崩溃、机器是否明显卡顿。


In [ ]:
for path in [
    "reports/stage5_dpo_plan.md",
    "reports/stage5a_dpo_tiny_data_report.md",
    "reports/stage5b_tiny_dpo_smoke_report.md",
    "reports/stage5c_tiny_dpo_behavior_report.md",
    "reports/stage5_dpo_revision_loop_report.md",
    "reports/stage5_candidate_dpo_v4_v5_report.md",
    "reports/stage5g_naive_dpo_v6_report.md",
    "reports/stage5_structured_behavior_score_report.md",
    "reports/stage5h_prompt7_data_and_eval_design.md",
    "reports/compare_outputs_four_way_dpo_tiny.jsonl",
    "reports/compare_outputs_four_way_dpo_tiny_v2.jsonl",
    "reports/compare_outputs_four_way_dpo_tiny_v3.jsonl",
    "reports/compare_outputs_four_way_dpo_candidate_v4.jsonl",
    "reports/compare_outputs_four_way_dpo_candidate_v5.jsonl",
    "reports/compare_outputs_four_way_dpo_naive_v6.jsonl",
    "reports/vram_and_dpo_plan.md",
    "configs/dpo_qwen05b.yaml",
    "configs/dpo_qwen05b_v4.yaml",
    "configs/dpo_qwen05b_v5.yaml",
    "configs/dpo_qwen05b_v6_naive.yaml",
    "data/processed/dpo_tiny_train.jsonl",
    "data/processed/dpo_candidate_train.jsonl",
    "data/processed/dpo_candidate_v5_train.jsonl",
    "data/processed/dpo_larger_v6_train.jsonl",
    "data/processed/dpo_stage5h_prompt7_train.jsonl",
    "data/processed/dpo_stage5h_prompt7_eval.jsonl",
    "data/samples/custom_technical_prompts_expanded_stage5h.jsonl",
]:
    print("\n" + "=" * 80)
    print(path)
    print("=" * 80)
    print(Path(path).read_text(encoding="utf-8")[:2200])


## 15. Stage 5H-5P：prompt 7 数据、expanded gate 和修复探针

Stage 5H 的目标是先把 prompt 7 的数据和评测尺子做强，而不是直接继续加 DPO step。

已生成：

- `data/processed/dpo_stage5h_prompt7_train.jsonl`：278 条 preference train，其中新增 72 条 prompt 7 修复 pair。
- `data/processed/dpo_stage5h_prompt7_eval.jsonl`：55 条 held-out eval，其中新增 24 条 prompt 7 eval pair。
- `data/samples/custom_technical_prompts_expanded_stage5h.jsonl`：24 题 expanded behavior suite，保留原始 8 题，新增 12 条 prompt 7 held-out 改写和 4 条 replay holdout。
- `scripts/score_expanded_behavior_outputs.py`：按 `prompt_area` 元数据评分，适合 expanded suite。

后续 Stage 5I-5P 已完成：

- Stage 5I：DPO v6 expanded gate 只有 7/24，loss-vs-behavior 0/13。
- Stage 5J：DPO v7 preference eval accuracy 1.0，但固定 prompt 仍 7/8，prompt 7 失败。
- Stage 5K：直接 SFT repair 从 SFT v3 出发，旧题严重回归，固定评分 1/8。
- Stage 5M：DPO v8 从 v6 做 exact-failure DPO，preference eval accuracy 1.0，但仍 7/8。
- Stage 5N：从 v6 做 micro-SFT，旧题稳定，仍 7/8，prompt 7 失败。
- Stage 5O：exact prompt 7 SFT 让 prompt 7 通过，但旧题回归到 4/8。
- Stage 5P：半 epoch 平衡探针仍未找到稳定点，固定评分 6/8。

结论：不要继续盲目加 DPO/SFT step。当前保守推荐 checkpoint 仍是 `outputs/sft_lora_qwen05b_custom_v3_from_v1_patch`，最好 DPO artifact 仍是 `outputs/dpo_lora_qwen05b_naive_v6`。


In [ ]:
for path in [
    "reports/stage5h_prompt7_data_and_eval_design.md",
    "data/processed/dpo_stage5h_prompt7_train.jsonl",
    "data/processed/dpo_stage5h_prompt7_eval.jsonl",
    "data/samples/custom_technical_prompts_expanded_stage5h.jsonl",
]:
    print("\n" + "=" * 80)
    print(path)
    print("=" * 80)
    rows = [line for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
    if path.endswith(".jsonl"):
        print("rows:", len(rows))
        for line in rows[:3]:
            item = json.loads(line)
            print(json.dumps(item, ensure_ascii=False)[:500])
    else:
        print(Path(path).read_text(encoding="utf-8")[:2200])


## 16. 当前验收清单

- Stage 0/1：环境和 base 推理完成。
- Stage 2A：公开中文 SFT 数据完成。
- Stage 3A：public LoRA SFT 完成。
- Stage 4A：base vs public-SFT 对比完成，结论是没有修正目标技术概念。
- Stage 2B：自采集技术数据完成并多轮修订，当前 142 train / 15 eval，另有 28 条 Stage 2B.3 focused patch。
- Stage 3B：custom-data LoRA SFT v1 完成，输出 `outputs/sft_lora_qwen05b_custom`。
- Stage 4B：三方对比完成，v1 改善 6/8 个固定技术 prompt。
- Stage 2B.2：badcase patch 完成；v2 从头训练回归，v3 低学习率续训保留或改善 7/8 个 prompt。
- Stage 2B.3：DPO 前 SFT gate 完成；v4/v5/v6 都不推荐，当前 best 仍是 v3。
- Stage 5A：tiny DPO preference 数据已完成，`data/processed/dpo_tiny_train.jsonl` 共 33 pair。
- Stage 5B：tiny DPO smoke test 已完成，输出 `outputs/dpo_lora_qwen05b_tiny`，无 OOM/原生崩溃，adapter 可加载。
- Stage 5C：固定 prompt 行为对比已完成，但行为 gate 未通过；不要直接扩大 DPO。
- Stage 5 v2/v3 修订循环：硬件继续通过，但行为仍不过 gate；v3 DPO 甚至明显回归。
- Stage 5 candidate v4/v5：candidate-derived 数据和 focused exact-failure 数据都跑通；v4/v5 仍然只有 6/8，通过不了 prompt 4 和 prompt 7。
- Stage 5 larger naive v6：192 train / 24 eval，separate reference，无 OOM；通过 7/8，是目前最好 DPO 候选，但 prompt 7 仍未过。
- Stage 5H：prompt 7 数据/eval 设计完成，已生成 278 train / 55 eval 和 24 题 expanded behavior suite。
- Stage 5I：v6 跑 expanded gate 后只通过 7/24，loss-vs-behavior 0/13，说明 8 题固定集之外仍有明显泛化缺口。
- Stage 5J/M：DPO v7/v8 的 preference eval accuracy 都到 1.0，但固定 prompt 仍是 7/8，prompt 7 未过。
- Stage 5K/N/O/P：直接 SFT 探针证明 prompt 7 可以被强行修好，但会带来旧题回归；Stage 5N 最稳但仍 7/8，Stage 5O prompt 7 过了却掉到 4/8，Stage 5P 也没找到稳定平衡。
- Stage 5 结构化行为评分：没有新的 DPO/SFT repair adapter 被接受；保守推荐 checkpoint 仍是 `outputs/sft_lora_qwen05b_custom_v3_from_v1_patch`，最佳 DPO artifact 仍是 `outputs/dpo_lora_qwen05b_naive_v6`。

后续如果继续推进，不建议只继续加 DPO step；应先设计更宽的 prompt 7 curriculum 和更强 replay。更直接的下一阶段是 Stage 6：把 loss / preference accuracy 不能替代 behavior gate 的复盘整理成面试叙事、before/after 示例和最终报告。这个文件的作用是把项目从“命令集合”变成一条能逐格运行、逐步理解的学习路线。

## 17. 下一次新聊天建议

新开空聊天时，建议先让 Codex 阅读交接文件，而不是直接继续训练：

```text
请先阅读这个项目：
D:\coding\qwen lorar sft\qwen-local-lora-sft-dpo

请重点阅读：
reports/next_chat_handoff_stage5g.md
reports/project_context_for_next_chat.md
reports/stage5g_naive_dpo_v6_report.md
reports/stage5j_to_5p_prompt7_repair_report.md
reports/stage5_structured_behavior_score_report.md
reports/stage5_dpo_plan.md
PROJECT_RUNBOOK.md
README.zh-CN.md
README.md
notebooks/04_full_pipeline_learning.ipynb

读完后，请先用中文总结当前状态、推荐 checkpoint、最好 DPO 候选、剩余问题和下一阶段计划。重点复盘 Stage 5I-5P 为什么 loss / preference accuracy 不能替代 behavior gate。不要继续盲目加 DPO step；如果要恢复训练，先设计更宽的 prompt 7 curriculum 和回归保护。
```

下一阶段的核心不是证明机器能跑 DPO；v6/v7/v8 已经证明了。核心是把训练指标、偏好指标和行为验收拆开讲清楚。
